# UAVIDS-2025 — estabilidade por sementes em S2

Esta rodada mantém dados, folds, atributos e hiperparâmetros fixos e varia somente a semente de ajuste. O objetivo é verificar se a incerteza algorítmica é relevante diante da heterogeneidade entre grupos de origem.


## Protocolo

Foram usadas três sementes para XGBoost e Random Forest nos cinco folds externos S2. As configurações vieram da rodada aninhada v2 e estão congeladas em [`stability_s2_v3.json`](../../configs/stability_s2_v3.json). As sementes são repetições do algoritmo, não novos datasets.


In [1]:
from pathlib import Path
import json
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "research":
    project_root = project_root.parents[1]
elif project_root.name == "notebooks":
    project_root = project_root.parent

result_dir = project_root / "results" / "stability_s2_v3"
report_dir = project_root / "reports" / "stability_s2_v3"
manifest = json.loads((result_dir / "experiment_manifest.json").read_text("utf-8"))
print(json.dumps(manifest, indent=2, ensure_ascii=False))


{
  "experiment_id": "stability_s2_v3",
  "status": "exploratory_seed_stability_after_nested_tuning",
  "config_sha256": "39cbec00e0e0af0f33c8e1df049d3ef2d18fd07222fa23f1c8f6ad6de72e209d",
  "dataset_sha256": "d50d339f68be7b23f0bf089dd438b20a1835c13182d8641220538121440164d0",
  "split_sha256": "352d1966e2dda8060a7a59d69eafa49c461e4c2415db292319e0e1526d2996cc",
  "completed_jobs": 30,
  "expected_jobs": 30,
  "complete": true,
  "environment": {
    "python": "3.12.10",
    "platform": "Windows-11-10.0.26200-SP0",
    "numpy": "1.26.4",
    "pandas": "2.2.3",
    "scikit_learn": "1.5.2",
    "xgboost": "2.0.3",
    "processor": "AMD64 Family 25 Model 33 Stepping 2, AuthenticAMD",
    "logical_cpu_count": 12,
    "threads_allowed_per_fit": 4,
    "accelerator": "none_used"
  },
  "class_order": [
    "Normal Traffic",
    "Blackhole Attack",
    "Flooding Attack",
    "Sybil Attack",
    "Wormhole Attack"
  ],
  "prediction_schema": [
    "row_id",
    "flow_id",
    "protocol",
    "fol

## Métricas OOF por semente

In [2]:
pooled_metrics = pd.read_csv(report_dir / "pooled_metrics_by_seed.csv")
print(pooled_metrics.round(6).to_string(index=False))


        model     seed  f1_macro  accuracy  log_loss  false_alarm_rate  missed_attack_rate
random_forest 20260907  0.951614  0.949407  0.117543          0.010049            0.003115
random_forest 20260908  0.951787  0.949505  0.118524          0.009934            0.003156
random_forest 20260909  0.952225  0.949914  0.117270          0.009781            0.003094
      xgboost 20260907  0.952666  0.950749  0.108483          0.008826            0.002927
      xgboost 20260908  0.952293  0.950373  0.108791          0.008979            0.002854
      xgboost 20260909  0.952947  0.950995  0.108686          0.008788            0.002990


## Decomposição descritiva da variação

O desvio entre sementes é calculado dentro de cada fold. O desvio entre folds usa a média das três sementes de cada fold. Essa separação evita misturar duas fontes de variação com interpretações diferentes.


In [3]:
variance = pd.read_csv(report_dir / "variance_decomposition.csv")
print(variance.round(6).to_string(index=False))


        model  overall_fold_seed_mean  mean_within_fold_seed_std  maximum_within_fold_seed_std  between_fold_std_of_seed_means  between_fold_range_of_seed_means
random_forest                0.950669                   0.000395                      0.000957                        0.005530                          0.013216
      xgboost                0.951650                   0.000479                      0.000657                        0.004584                          0.010220


![Variação entre folds e sementes](../../reports/stability_s2_v3/fold_and_seed_variation.png)

## Diferença pareada entre modelos

In [4]:
paired = pd.read_csv(report_dir / "paired_model_differences.csv")
print(paired.round(6).to_string(index=False))


 fold     seed  random_forest  xgboost  xgboost_minus_random_forest
    0 20260907       0.947586 0.948015                     0.000429
    0 20260908       0.947686 0.947615                    -0.000072
    0 20260909       0.948027 0.947378                    -0.000649
    1 20260907       0.952845 0.951935                    -0.000910
    1 20260908       0.953196 0.951402                    -0.001794
    1 20260909       0.952930 0.952176                    -0.000754
    2 20260907       0.954767 0.955278                     0.000511
    2 20260908       0.955379 0.955241                    -0.000138
    2 20260909       0.956644 0.956397                    -0.000247
    3 20260907       0.942543 0.945997                     0.003455
    3 20260908       0.942432 0.946233                     0.003801
    3 20260909       0.942168 0.947094                     0.004926
    4 20260907       0.954330 0.956898                     0.002567
    4 20260908       0.954414 0.956149          

## Interpretação

- O desvio médio entre sementes foi **0.000395** para RF e **0.000479** para XGBoost.
- O desvio entre folds foi **0.005530** para RF e **0.004584** para XGBoost.
- XGBoost superou RF em **8/15** pares, com diferença média **+0.000981**.

A origem mantida fora do treino explica mais variação que a semente. A evidência não sustenta dizer que XGBoost domina RF em qualquer origem; sustenta apenas uma vantagem média pequena nesta população S2, combinada a menor tamanho serializado e menor taxa de falso alarme.

O próximo passo operacional pode congelar ambos os modelos e medir batch 1, pipeline e requisição. Esses benchmarks não corrigirão a ausência de novas simulações externas.
